# Gemma 4 ESCO PEFT Fine-Tuning & Measurement Pipeline

Welcome to the official repository notebook for fine-tuning **Gemma 4** on **ESCO v1.2.1** occupational skill mappings using parameter-efficient fine-tuning (QLoRA) with **Unsloth** and Hugging Face PEFT.

**Author:** Mohammadreza A. Fard  
**Repository:** [github.com/mazafard/esco-gemma4-pipeline](https://github.com/mazafard/esco-gemma4-pipeline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mazafard/esco-gemma4-pipeline/blob/main/esco_gemma4_pipeline.ipynb)

## 🛠️ Step 1: Check GPU Runtime
To use high-performance Unsloth training, select a GPU runtime (T4, L4, or A100) under **Runtime > Change runtime type**.

In [ ]:
!nvidia-smi

## 📂 Step 2: Clone Repository & Avoid Folder Nesting
This cell clones your repository. If the folder already exists, it resets your working directory and pulls the latest changes (preventing nested directory issues).

In [ ]:
import os

# Prevent nested directories by always starting from /content
%cd /content

if not os.path.exists('/content/esco-gemma4-pipeline'):
    !git clone https://github.com/mazafard/esco-gemma4-pipeline.git

%cd /content/esco-gemma4-pipeline
!git pull

## 📊 Step 3: Provide raw ESCO CSV files
Since the raw **`ESCO dataset - v1.2.1 - classification - en - csv`** folder contains large files, it is excluded in `.gitignore` and is not included when cloning the repository from GitHub.

You have **two options** to make the raw CSV data available in Google Colab:

### Option A: Drag & Drop (Easiest)
1. In the Google Colab left sidebar, click the **Files (Folder)** icon.
2. Navigate to `/content/esco-gemma4-pipeline`.
3. Drag and drop your local **`ESCO dataset - v1.2.1 - classification - en - csv`** folder directly into the `esco-gemma4-pipeline` folder.

### Option B: Mount Google Drive
If you have uploaded the ESCO CSV folder to your Google Drive, run the cell below to mount your drive and link the folder directly:

In [ ]:
# Run this cell ONLY if using Option B (Google Drive)
from google.colab import drive
import os

drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/ESCO dataset - v1.2.1 - classification - en - csv"
target_path = "/content/esco-gemma4-pipeline/ESCO dataset - v1.2.1 - classification - en - csv"

if os.path.exists(drive_path):
    if not os.path.exists(target_path):
        os.symlink(drive_path, target_path)
        print("✅ Successfully linked ESCO CSV dataset from Google Drive!")
else:
    print("⚠️ ESCO CSV folder not found in your Google Drive root. Please verify the folder name matches exactly.")

## 📦 Step 4: Install Dependencies
We will install the optimized Unsloth framework and requirements.

In [ ]:
# Install Unsloth & standard ML packages optimized for Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -r requirements.txt
!pip install -U huggingface_hub

## 🔑 Step 5: Hugging Face Authentication
Provide your Hugging Face write token to publish the final merged model adapter weights.

In [ ]:
import os
from google.colab import userdata

# Set your HF Token. You can add it securely in Colab Secrets as 'HF_TOKEN' or enter it below:
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_WRITE_TOKEN_HERE"

## 🚀 Step 6a: Phase 1 - Data Preparation
Ingests the ESCO dataset, maps skills to occupations, and structures the prompt templates.

In [ ]:
!python pipeline.py --stage prep

## ⬇️ Step 6b: Pre-Download Base Model
Downloads the base model into Colab's cache separately so that the fine-tuning step is cleaner and doesn't time out.

In [ ]:
# Executes Unsloth's caching mechanism locally
!python pipeline.py --stage download


## 🧠 Step 6c: Phase 2 - Parameter-Efficient Fine-Tuning (PEFT)
Loads the base Gemma 4 model into GPU VRAM using Unsloth 4-bit quantization and begins QLoRA training.

In [ ]:
!python pipeline.py --stage train

## ☁️ Step 6d: Phase 3 - Merge Adapters & Push to Hugging Face
Merges the generated QLoRA adapters with the base model and securely uploads it to your Hugging Face repository.

In [ ]:
# Make sure to set your correct Hugging Face repository name here:
!python pipeline.py --stage merge --hf_repo "mazafard/esco-gemma4-pipeline"

## 📝 Step 6e: Phase 4 - Generate Academic Journal
Compiles all runtime telemetry (loss, VRAM usage, Precision/Recall) into an academic markdown report.

In [ ]:
!python pipeline.py --stage journal

## 📄 Step 7: View Compiled Academic Journal

In [ ]:
# Display the generated research methodology paper directly in the notebook
from IPython.display import Markdown
import os

journal_path = "artifacts/experimental_journal.md"
if os.path.exists(journal_path):
    with open(journal_path, "r") as f:
        journal_content = f.read()
    display(Markdown(journal_content))
else:
    print("⚠️ Academic Journal file not found!")
    print("This occurs when the pipeline halts early due to a previous phase's error (e.g. missing HF token).")
    print("Please check the log output in the cell above or logs/pipeline.log for details.")